In [11]:
!pip install langchain_community

In [22]:
#pdf_name ='326113641__0.pdf'
#pdf_name ='InvoiceRE-202659666.pdf'
pdf_name ='SalesInvoice.pdf'

In [23]:
!pip install pypdf
!pip install -U langchain-text-splitters

In [43]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

reader = PdfReader(pdf_name)
page = reader.pages[0]
text = page.extract_text()
print("Original text length:", len(text))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True,
)

texts = text_splitter.create_documents([text])

print(f"Split into {len(texts)} documents")
print("First document content (truncated): ") # Added space for clarity
print(texts[0].page_content[:500]) # Print first 500 characters of the first chunk

Original text length: 2041
Split into 3 documents
First document content (truncated): 
Facture
Invoive No   
2607361
Billing Address
ZAKI ASIAN FOODS
         
ZAKI 
EXOTIC CITY SRL
Universitätsstr ,40
Avenue de l'expansion 1
Duisburg, 47051
ALLEUR
Bill-to Customer 
2020564
 Enterprise No.
  
BE 0882.540.147
Enterprise No.
 
  Phone No.
VATRegistrationNo
352116861
No.Telephone No.
 E-mail   
             
info@exoticcity.be
Invoice No.
2607361
  HomePage 
https://exoticcity.be
 Document Date
03-05-2026
  Bank
BELFIUS
Posting Date
03-05-2026
 SWIFT Code
GKCCBEBB
Due Date
10-05-2026


In [25]:
# Install the Qdrant client library
!pip install qdrant-client

In [26]:
from qdrant_client import QdrantClient
from google.colab import userdata

# Retrieve API key and host from Colab secrets
# IMPORTANT: Ensure that the values in Colab secrets (e.g., for QDRANT_HOST) are stored without extra quotes.
# For example, store `https://your-host.qdrant.io` NOT `"https://your-host.qdrant.io"`

# NOTE: QDRANT_API_KEY should be retrieved from a dedicated secret for the API key.
# If you named your API key secret 'qdrant_api_key', use userdata.get('qdrant_api_key')
# Assuming 'qdrant' was intended for the API key based on previous context.
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
QDRANT_HOST = userdata.get('QDRANT_HOST') # Ensure this secret's value is just the URL, no quotes.

# Initialize Qdrant client
# The Qdrant client expects 'url' parameter if the address includes http(s)://
client = QdrantClient(
    url=QDRANT_HOST,
    api_key=QDRANT_API_KEY,
)

print("\n--- Performing Qdrant Connection Check ---")

try:
    # Attempt to list collections as a simple connection test
    collections = client.get_collections()
    print("Successfully connected to Qdrant! Existing collections:")
    for collection in collections.collections:
        print(f"- {collection.name}")
except Exception as e:
    print(f"Error connecting to Qdrant or listing collections: {e}")
    print("Please check your QDRANT_HOST, QDRANT_API_KEY, and network connection, and ensure secrets are formatted correctly (no extra quotes).")

print("--- Connection Check Complete ---")

print("Qdrant client initialized successfully.")
# You can now interact with your Qdrant database, e.g., create a collection:
# client.recreate_collection(collection_name="my_collection", vectors_config=VectorParams(size=1536, distance=Distance.COSINE))
# print("Collection 'my_collection' created (if it didn't exist).")


--- Performing Qdrant Connection Check ---
Successfully connected to Qdrant! Existing collections:
- pdf_document_embeddings
--- Connection Check Complete ---
Qdrant client initialized successfully.


In [ ]:
!pip install sentence-transformers


In [44]:
from sentence_transformers import SentenceTransformer
from qdrant_client.http.models import Distance, VectorParams

# Define the embedding model
embedding_model_name = "all-MiniLM-L6-v2" # A good, lightweight general-purpose model
embedding_dimension = 384 # This model outputs 384-dimensional vectors

print(f"Loading Sentence Transformer model: {embedding_model_name}...")
model = SentenceTransformer(embedding_model_name)
print("Model loaded.")

# Generate embeddings for the text chunks
print("Generating embeddings for text chunks...")
# 'texts' is a list of Document objects from Langchain, extract page_content
text_contents = [doc.page_content for doc in texts]
embeddings = model.encode(text_contents, show_progress_bar=True)
print(f"Generated {len(embeddings)} embeddings of dimension {embeddings.shape[1]}.")

Loading Sentence Transformer model: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded.
Generating embeddings for text chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated 3 embeddings of dimension 384.


In [45]:
collection_name = "pdf_document_embeddings"

# Create a new Qdrant collection or recreate if it already exists
print(f"Creating/recreating Qdrant collection: '{collection_name}'...")
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=embedding_dimension, distance=Distance.COSINE),
)
print(f"Collection '{collection_name}' ready.")

# Prepare points for upserting into Qdrant
points = []
for i, (embedding, text_content) in enumerate(zip(embeddings, text_contents)):
    points.append({
        "id": i, # Using index as ID
        "vector": embedding.tolist(), # Convert numpy array to list
        "payload": {"text": text_content}
    })

print(f"Preparing to upsert {len(points)} points into collection '{collection_name}'...") # Added confirmation print

# Upsert points to Qdrant
print(f"Upserting {len(points)} points into collection '{collection_name}'...")
client.upsert(
    collection_name=collection_name,
    wait=True, # Wait for the operation to complete
    points=points
)
print(f"Successfully added {len(points)} points to Qdrant collection '{collection_name}'.")

Creating/recreating Qdrant collection: 'pdf_document_embeddings'...


/tmp/ipykernel_21225/2154921251.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Collection 'pdf_document_embeddings' ready.
Preparing to upsert 3 points into collection 'pdf_document_embeddings'...
Upserting 3 points into collection 'pdf_document_embeddings'...
Successfully added 3 points to Qdrant collection 'pdf_document_embeddings'.


-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------end of vector store operation

In [31]:
!pip install -U langchain
!pip install -qU langchain-qdrant
!pip install -U langchain-google-genai
!pip install -U langchain-huggingface

### Creating a Langchain Agent with Google Gemini and Qdrant

Now that Qdrant is connected and populated with embeddings, let's create a Langchain agent. This agent will use Google Gemini as its language model and will have access to your Qdrant vector store as a tool to answer questions by retrieving relevant information from your PDF documents.

In [49]:
import langchain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool
from langchain_qdrant import QdrantVectorStore
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from qdrant_client.http.models import Distance, VectorParams

# Set up Google Gemini API key from Colab secrets
# Ensure you have a secret named 'GOOGLE_API_KEY' in Colab with your Gemini API key
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Initialize Google Gemini model
# Updated model name based on available models list
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY)

# Create a Langchain-compatible embeddings object
embeddings_model = HuggingFaceEmbeddings(model_name=embedding_model_name)

# Create a Qdrant vector store from the existing client and collection
# Specify content_payload_key to correctly extract text from Qdrant's payload
qdrant_vectorstore = QdrantVectorStore(client=client, collection_name=collection_name, embedding=embeddings_model, content_payload_key="text")

# Define a tool to query the Qdrant vector store
@tool
def search_qdrant(query: str) -> str:
    """Searches the Qdrant vector store for documents similar to the query."""
    # Use the vector store's built-in similarity search
    docs = qdrant_vectorstore.similarity_search(query, k=1)
    if docs:
        return docs[0].page_content
    return "No relevant documents found in Qdrant."

# Define the tools available to the agent
tools = [search_qdrant]

# Extract the system message content as a string
system_message_content = "You are a helpful assistant. Use the available tools to answer questions about the provided documents. If the question can be answered by searching Qdrant, use the 'search_qdrant' tool."

# Create the full prompt template, if needed for agent invocation later
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_message_content),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

# Create the agent, passing the system message content as a string
graph = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_message_content,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [50]:
question = "what is the invoice number"
question = "what of the inovice have cheaper same items then other vendors"

In [51]:
inputs = {"messages": [{"role": "user", "content": question}]}

# Now that the LLM model name has been updated in cell 2f1d865e,
# and Qdrant is confirmed to be refreshed, we can stream the graph to get the agent's response.
for chunk in graph.stream(inputs, stream_mode="updates"):
    print(chunk)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_qdrant', 'arguments': '{"query": "cheaper same items then other vendors in the invoice"}'}, '__gemini_function_call_thought_signatures__': {'274ed4ae-d081-4140-837b-8c03028f1633': 'CoIDARFNMg+VPXsVjxVCnV96QcehJaRf02LOmI2jzI84dvSBtHoklnZvf8cp35KaHLjPskc6vPLtdkXeWEM+u+4UQZyzldTci1BKNUCX9sI4TENxseuCGf+Y1FAxG1pjnG9RR43TWwFLDbMJdIjF6LerdKDlLkuLbPczEi31RSyd69zqY6q2C3KDn5FTQJ+p2Wuo2Lc1jvsUMcvU81d2wwI6WhFjSnAq9/Ox7x38Hdgt+ZK8hVFrlJzAdkzVFD8fSRBekadpzhdOLBpHx/8pnI7Lr16aU8r94XJUTmDDa2O6kUqpi6NBHJZWBOefguL/kZPpgXd6fr9fHNDxPnS88+uAlQ0avXrsz0Hl65iDm75zObob86afE184JQpErlzfr/JaE9ik1sw1kToYwu1ijh+FKw1F01L4AflyeCwqU5ZA4lggsbfEuTEL1/F+6rdmbhTEbJHG8+yN32n+tglbsbxsqq0zHv+FV8pMg1F86jY8ZSY8hxCJbworLk/neIu/DE1Iugw='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fe363-e7c0-7511-b160-fe8b999cc414-0', tool_ca

In [61]:
print(chunk['model']['messages'][0].content[0]['text'])

The provided invoice does not contain information about other vendors to compare prices for the same items. It only lists items from "EXOTIC CITY SRL". Therefore, I cannot identify if there are cheaper items from other vendors based on this document.
